# Construindo Aplicações de Geração de Imagens (Azure OpenAI)

Há mais nos LLMs do que geração de texto. Você também pode gerar imagens a partir de descrições em texto. Imagens como uma modalidade são úteis em MedTech, arquitetura, turismo, desenvolvimento de jogos, marketing e muito mais. Nesta lição, trabalhamos com os modelos **GPT Image** atuais no Microsoft Foundry.

## Objetivos de aprendizagem

Ao final desta lição você será capaz de:

- Explicar o que é geração de imagens e onde é útil.
- Entender a família de modelos `gpt-image` e como ela difere dos modelos legados DALL·E.
- Construir uma aplicação de geração de imagens e editar imagens.

## O que é geração de imagens?

Modelos de geração de imagens criam imagens a partir de um prompt em texto. Modelos modernos, como o `gpt-image`, aprendem a relação entre texto e imagens durante o treinamento, e então iterativamente transformam ruído aleatório em uma imagem que corresponde ao seu prompt.

Duas famílias de modelos de imagem bem conhecidas são:

- **`gpt-image` (OpenAI)** — a geração atual usada nesta lição. Suporta geração de imagem a partir de texto e edição de imagem (restauração com máscara).
- **Midjourney** — um modelo popular de terceiros com seu próprio serviço e fluxo de trabalho baseado em Discord.

> Os modelos OpenAI de imagem mais antigos — **DALL·E 2** e **DALL·E 3** — são legados. DALL·E 3 não está mais disponível para novas implantações, e recursos como `create_variation` existiam apenas no DALL·E 2. Use os modelos `gpt-image` para novas aplicações.

No Microsoft Foundry, o **`gpt-image-2`** é o modelo de imagem mais recente e com mais recursos, sendo o padrão recomendado. `gpt-image-1.5` e `gpt-image-1-mini` também estão geralmente disponíveis.

> **Importante:** modelos `gpt-image` retornam a imagem gerada em **base64** (`b64_json`), não como uma URL. Seu código decodifica a string base64 para bytes e salva — não há URL da imagem para baixar.


## Construindo sua primeira aplicação de geração de imagens

Então, o que é preciso para construir uma aplicação de geração de imagens? Você precisa das seguintes bibliotecas:

- **python-dotenv**, é altamente recomendado usar esta biblioteca para manter seus segredos em um arquivo *.env* separado do código.
- **openai**, esta biblioteca é o que você usará para interagir com a API do OpenAI.
- **pillow**, para trabalhar com imagens em Python.

Se ainda não fez isso, siga as instruções na página [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) para criar um recurso e modelo Microsoft Foundry. Selecione **gpt-image-2** como modelo (o modelo mais recente de imagem Azure OpenAI; DALL·E é legado).

1. Crie um arquivo *.env* com o seguinte conteúdo:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Localize estas informações no portal Microsoft Foundry para seu recurso na seção "Deployments".



1. Colete as bibliotecas acima em um arquivo chamado *requirements.txt* assim:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Em seguida, crie um ambiente virtual e instale as bibliotecas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para Windows, use os seguintes comandos para criar e ativar seu ambiente virtual:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Adicione o seguinte código no arquivo chamado *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # configurar o cliente Azure OpenAI (Microsoft Foundry).
    # Modelos de imagem precisam de uma versão recente da API — verifique a documentação do Foundry para a versão exigida pelo seu modelo.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Crie uma imagem usando a API de geração de imagens
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Insira seu texto de prompt aqui
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # por exemplo, "gpt-image-2"
        )
        # Defina o diretório para armazenar a imagem
        image_dir = os.path.join(os.curdir, 'images')

        # Se o diretório não existir, crie-o
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicialize o caminho da imagem (observe que o tipo de arquivo deve ser png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Modelos gpt-image retornam a imagem como base64 (b64_json), não uma URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Exibir a imagem no visualizador de imagens padrão
        image = Image.open(image_path)
        image.show()

    # capturar exceções
    except BadRequestError as err:
        print(err)

    ```

Vamos explicar este código:

- Primeiro, importamos as bibliotecas que precisamos, incluindo a biblioteca OpenAI, a biblioteca dotenv, o módulo base64, e a biblioteca Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Em seguida, carregamos as variáveis de ambiente do arquivo *.env*.

    ```python
    # importar dotenv
    dotenv.load_dotenv()
    ```

- Depois disso, configuramos o cliente Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Em seguida, geramos a imagem:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Digite seu texto de prompt aqui
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Modelos `gpt-image` retornam a imagem como uma string **base64** em `data[0].b64_json`. Decodificamos para bytes e escrevemos em um arquivo — não há URL para download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Por fim, abrimos a imagem e usamos o visualizador de imagens padrão para exibí-la:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Mais detalhes sobre a geração da imagem

Vamos observar os parâmetros de `images.generate`:

- **prompt**, é a instrução em texto usada para gerar a imagem. Aqui é "Coelho em cavalo, segurando um pirulito, em um prado nebuloso onde crescem narcisos".
- **size**, é o tamanho da imagem gerada (por exemplo `1024x1024`, `1536x1024`, `1024x1536`, ou `"auto"`).
- **n**, é o número de imagens geradas. Aqui geramos uma.
- **model**, é o nome do seu modelo de imagem implantado (por exemplo `gpt-image-2`).

> Modelos de imagem não usam o parâmetro `temperature` — isso é um controle para geração de texto. Para obter variedade, chame a API novamente; para reduzir a variedade, torne seu prompt mais específico.

## Capacidades adicionais da geração de imagem

Você viu como gerar uma imagem com poucas linhas de Python. Modelos `gpt-image` também podem **editar** uma imagem existente. Fornecendo uma imagem, uma **máscara** opcional (que marca a área a ser alterada) e um prompt, você pode modificar parte de uma imagem — por exemplo, adicionar um chapéu no nosso coelho.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# as edições também são retornadas como base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

A imagem base contém apenas o coelho; a imagem final tem o chapéu no coelho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
